# Part 2 — Pointwise VS Pairwise Learning with Ranking Metric

This notebook uses **MetadataMLP** directly and compares two training strategies on the same model:
- standard pointwise regression training (`mse`)
- pairwise ranking training with **BPR loss** (`bpr`)

Evaluation focuses on ranking quality from the 0–5 ratings by treating ratings `>= 4.0` as relevant.


In [ ]:
import copy
import math
import os
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
os.makedirs('./checkpoints', exist_ok=True)


Using device: cuda


In [ ]:
METADATA_CATEGORICAL_COLS = ['beer/style', 'beer/brewerId']
METADATA_NUMERIC_COLS = ['beer/ABV']
USER_COL = 'review/profileName'
ITEM_COL = 'beer/beerId'
RATING_COL = 'review/overall'

RAW_TRAIN = pd.read_json('./data/beeradvocate_train.json', lines=True)
RAW_VAL = pd.read_json('./data/beeradvocate_val.json', lines=True)
RAW_TEST = pd.read_json('./data/beeradvocate_test.json', lines=True)

required_cols = [USER_COL, ITEM_COL, RATING_COL] + METADATA_CATEGORICAL_COLS + METADATA_NUMERIC_COLS
missing = [c for c in required_cols if c not in RAW_TRAIN.columns]
if missing:
    raise ValueError(f'Missing required columns in training data: {missing}')

def prepare_split(df):
    split = df[required_cols].copy()
    split = split.dropna(subset=[USER_COL, ITEM_COL, RATING_COL])
    for col in METADATA_CATEGORICAL_COLS:
        split[col] = split[col].fillna('<UNK>').astype(str)
    for col in METADATA_NUMERIC_COLS:
        split[col] = pd.to_numeric(split[col], errors='coerce')
    return split

train_df = prepare_split(RAW_TRAIN)
val_df = prepare_split(RAW_VAL)
test_df = prepare_split(RAW_TEST)

for col in METADATA_NUMERIC_COLS:
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    val_df[col] = val_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

user2idx = {u: i for i, u in enumerate(train_df[USER_COL].unique())}
item2idx = {it: i for i, it in enumerate(train_df[ITEM_COL].unique())}
unknown_user_idx = len(user2idx)
unknown_item_idx = len(item2idx)
num_users = len(user2idx) + 1
num_items = len(item2idx) + 1

categorical_maps = {}
categorical_sizes = {}
for col in METADATA_CATEGORICAL_COLS:
    values = sorted(train_df[col].unique().tolist())
    mapping = {v: i for i, v in enumerate(values)}
    categorical_maps[col] = mapping
    categorical_sizes[col] = len(mapping) + 1

numeric_stats = {}
for col in METADATA_NUMERIC_COLS:
    mean = float(train_df[col].mean())
    std = float(train_df[col].std())
    if std < 1e-8:
        std = 1.0
    numeric_stats[col] = (mean, std)


def encode_split(df):
    out = pd.DataFrame()
    out['user_idx'] = df[USER_COL].map(lambda x: user2idx.get(x, unknown_user_idx)).astype(np.int64)
    out['item_idx'] = df[ITEM_COL].map(lambda x: item2idx.get(x, unknown_item_idx)).astype(np.int64)
    out['rating'] = df[RATING_COL].astype(np.float32)

    for col in METADATA_CATEGORICAL_COLS:
        mapping = categorical_maps[col]
        out[f'{col}__idx'] = df[col].map(lambda x: mapping.get(x, len(mapping))).astype(np.int64)

    for col in METADATA_NUMERIC_COLS:
        mean, std = numeric_stats[col]
        out[f'{col}__num'] = ((df[col].astype(np.float32) - mean) / std).astype(np.float32)
    return out

train_encoded = encode_split(train_df)
val_encoded = encode_split(val_df)
test_encoded = encode_split(test_df)

print('Train encoded:', train_encoded.shape)
print('Val encoded:', val_encoded.shape)
print('Test encoded:', test_encoded.shape)
print('Users:', num_users, 'Items:', num_items)
print('Metadata categorical sizes:', categorical_sizes)


RANKING_POSITIVE_THRESHOLD = 4.0
RANKING_K = 10
RANKING_NUM_NEGATIVES = 100

user_seen_items = defaultdict(set)
for row in train_encoded[['user_idx', 'item_idx']].itertuples(index=False):
    user_seen_items[int(row.user_idx)].add(int(row.item_idx))


Train encoded: (1269292, 6)
Val encoded: (158661, 6)
Test encoded: (158661, 6)
Users: 27573 Items: 54912
Metadata categorical sizes: {'beer/style': 105, 'beer/brewerId': 5232}


In [ ]:
class MetadataRatingsDataset(Dataset):
    def __init__(self, df, categorical_cols, numeric_cols):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        self.categorical_cols = list(categorical_cols)
        self.numeric_cols = list(numeric_cols)
        self.cat_features = {
            col: torch.tensor(df[f'{col}__idx'].values, dtype=torch.long)
            for col in self.categorical_cols
        }
        self.num_features = {
            col: torch.tensor(df[f'{col}__num'].values, dtype=torch.float32)
            for col in self.numeric_cols
        }

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        cat = {col: values[idx] for col, values in self.cat_features.items()}
        num = {col: values[idx] for col, values in self.num_features.items()}
        return self.users[idx], self.items[idx], self.ratings[idx], cat, num


def metadata_collate_fn(batch):
    users, items, ratings, cat_features, num_features = zip(*batch)
    users = torch.stack(users)
    items = torch.stack(items)
    ratings = torch.stack(ratings)
    cat_batch = {
        col: torch.stack([features[col] for features in cat_features])
        for col in METADATA_CATEGORICAL_COLS
    }
    num_batch = {
        col: torch.stack([features[col] for features in num_features])
        for col in METADATA_NUMERIC_COLS
    }
    return users, items, ratings, cat_batch, num_batch


def make_metadata_loaders(batch_size=256):
    train_dataset = MetadataRatingsDataset(train_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
    val_dataset = MetadataRatingsDataset(val_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
    test_dataset = MetadataRatingsDataset(test_encoded, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=metadata_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=metadata_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=metadata_collate_fn)
    return train_loader, val_loader, test_loader


def move_feature_dict_to_device(feature_dict, device, kind='long'):
    dtype = torch.long if kind == 'long' else torch.float32
    return {k: v.to(device=device, dtype=dtype) for k, v in feature_dict.items()}


In [ ]:
class MetadataEncoder(nn.Module):
    def __init__(self, embedding_dim, categorical_sizes, numeric_cols):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.categorical_cols = list(categorical_sizes.keys())
        self.numeric_cols = list(numeric_cols)
        self.categorical_embeddings = nn.ModuleDict({
            col: nn.Embedding(size, embedding_dim) for col, size in categorical_sizes.items()
        })
        for emb in self.categorical_embeddings.values():
            nn.init.normal_(emb.weight, std=0.01)
        self.numeric_projection = nn.Linear(len(self.numeric_cols), embedding_dim) if self.numeric_cols else None
        self.output_projection = nn.Linear(
            embedding_dim * max(1, len(self.categorical_cols) + (1 if self.numeric_projection is not None else 0)),
            embedding_dim,
        )

    def forward(self, categorical_features, numeric_features):
        pieces = []
        for col in self.categorical_cols:
            pieces.append(self.categorical_embeddings[col](categorical_features[col]))
        if self.numeric_projection is not None:
            num_tensor = torch.stack([numeric_features[col] for col in self.numeric_cols], dim=1)
            pieces.append(self.numeric_projection(num_tensor))
        concat = torch.cat(pieces, dim=1)
        return self.output_projection(concat)


class MetadataMLP(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, hidden_dims, dropout, categorical_sizes, numeric_cols):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        self.metadata_encoder = MetadataEncoder(embedding_dim, categorical_sizes, numeric_cols)
        layers = []
        input_dim = embedding_dim * 3
        for h in hidden_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            input_dim = h
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(input_dim, 1)

    def forward(self, user_ids, item_ids, categorical_features, numeric_features):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)
        metadata_vec = self.metadata_encoder(categorical_features, numeric_features)
        x = torch.cat([user_vec, item_vec, metadata_vec], dim=1)
        return self.output_layer(self.mlp(x)).squeeze(1)


In [ ]:
# Build item-level metadata lookup from the training catalog so that sampled negative items
# use their own metadata rather than reusing the positive item's metadata.
item_metadata_source = (
    pd.concat([train_encoded, val_encoded, test_encoded], ignore_index=True)
    .sort_values('item_idx')
    .groupby('item_idx', as_index=True)
    .first()
)


def build_item_feature_tensors(item_ids, item_metadata_table, categorical_cols, numeric_cols, device):
    item_ids_cpu = item_ids.detach().cpu().tolist()
    rows = item_metadata_table.loc[item_ids_cpu]
    cat_features = {
        col: torch.tensor(rows[f'{col}__idx'].values, dtype=torch.long, device=device)
        for col in categorical_cols
    }
    num_features = {
        col: torch.tensor(rows[f'{col}__num'].values, dtype=torch.float32, device=device)
        for col in numeric_cols
    }
    return cat_features, num_features


def sample_negative_items(user_ids, user_seen_lookup, num_items):
    negatives = []
    for user_id in user_ids.detach().cpu().tolist():
        seen = user_seen_lookup.get(int(user_id), set())
        neg_item = random.randrange(num_items - 1)
        while neg_item in seen:
            neg_item = random.randrange(num_items - 1)
        negatives.append(neg_item)
    return torch.tensor(negatives, dtype=torch.long, device=user_ids.device)


def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()


def build_positive_lookup(df, threshold=4.0):
    positives = defaultdict(set)
    for row in df[['user_idx', 'item_idx', 'rating']].itertuples(index=False):
        if float(row.rating) >= threshold and int(row.user_idx) != unknown_user_idx and int(row.item_idx) != unknown_item_idx:
            positives[int(row.user_idx)].add(int(row.item_idx))
    return positives


def evaluate_rmse_mae(model, data_loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for users, items, ratings, cat_features, num_features in data_loader:
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)
            cat_features = move_feature_dict_to_device(cat_features, device, 'long')
            num_features = move_feature_dict_to_device(num_features, device, 'float')
            outputs = model(users, items, cat_features, num_features)
            preds.extend(outputs.detach().cpu().numpy().tolist())
            targets.extend(ratings.detach().cpu().numpy().tolist())
    rmse = float(np.sqrt(mean_squared_error(targets, preds)))
    mae = float(mean_absolute_error(targets, preds))
    return {'rmse': rmse, 'mae': mae}


def evaluate_ranking(model, eval_df, k=10, threshold=4.0, num_negatives=100, seed=42):
    model.eval()
    rng = random.Random(seed)
    positives_by_user = build_positive_lookup(eval_df, threshold)
    all_items = set(range(num_items - 1))
    hits, ndcgs, recalls = [], [], []

    with torch.no_grad():
        for user_idx, positive_items in positives_by_user.items():
            if not positive_items:
                continue
            seen_items = set(user_seen_items.get(user_idx, set()))
            candidates = list(all_items - seen_items - set(positive_items))
            if not candidates:
                continue
            if len(candidates) > num_negatives:
                candidates = rng.sample(candidates, num_negatives)
            candidate_items = list(positive_items) + candidates
            candidate_items = [it for it in candidate_items if it in item_metadata_source.index]
            if not candidate_items:
                continue
            user_tensor = torch.full((len(candidate_items),), user_idx, dtype=torch.long, device=device)
            item_tensor = torch.tensor(candidate_items, dtype=torch.long, device=device)
            cat_features, num_features = build_item_feature_tensors(
                item_tensor, item_metadata_source, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS, device
            )
            scores = model(user_tensor, item_tensor, cat_features, num_features).detach().cpu().numpy()
            ranked_items = [item for _, item in sorted(zip(scores, candidate_items), key=lambda x: x[0], reverse=True)]
            top_k = ranked_items[:k]
            num_hits = sum(1 for item in top_k if item in positive_items)
            hits.append(1.0 if num_hits > 0 else 0.0)
            recalls.append(num_hits / max(1, len(positive_items)))
            dcg = sum((1.0 / np.log2(rank + 1)) for rank, item in enumerate(top_k, start=1) if item in positive_items)
            ideal_hits = min(len(positive_items), k)
            idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        'num_users_evaluated': len(hits),
        f'hit_rate@{k}': float(np.mean(hits)) if hits else float('nan'),
        f'ndcg@{k}': float(np.mean(ndcgs)) if ndcgs else float('nan'),
        f'recall@{k}': float(np.mean(recalls)) if recalls else float('nan'),
    }


# For a fair head-to-head comparison, both training modes are selected by the same validation ranking metric.
def train_metadata_mlp(model, train_loader, epochs=8, lr=1e-3, weight_decay=1e-5, loss_type='mse'):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = copy.deepcopy(model.state_dict())
    best_metric = float('-inf')

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for users, items, ratings, cat_features, num_features in tqdm(train_loader, desc=f'{loss_type.upper()} epoch {epoch+1}/{epochs}', leave=False):
            users = users.to(device)
            items = items.to(device)
            ratings = ratings.to(device)
            cat_features = move_feature_dict_to_device(cat_features, device, 'long')
            num_features = move_feature_dict_to_device(num_features, device, 'float')
            optimizer.zero_grad()
            if loss_type == 'bpr':
                negative_items = sample_negative_items(users, user_seen_items, num_items)
                negative_cat_features, negative_num_features = build_item_feature_tensors(
                    negative_items, item_metadata_source, METADATA_CATEGORICAL_COLS, METADATA_NUMERIC_COLS, device
                )
                pos_scores = model(users, items, cat_features, num_features)
                neg_scores = model(users, negative_items, negative_cat_features, negative_num_features)
                loss = bpr_loss(pos_scores, neg_scores)
            else:
                outputs = model(users, items, cat_features, num_features)
                loss = criterion(outputs, ratings)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * users.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        val_metrics = evaluate_ranking(model, val_encoded, k=RANKING_K, threshold=RANKING_POSITIVE_THRESHOLD, num_negatives=RANKING_NUM_NEGATIVES)
        val_ndcg = val_metrics[f'ndcg@{RANKING_K}']
        print(f'Epoch {epoch+1}/{epochs} - train_loss={avg_loss:.4f} - val_ndcg@{RANKING_K}={val_ndcg:.4f}')

        if val_ndcg > best_metric:
            best_metric = val_ndcg
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return best_metric


CONFIG = {
    'embedding_dim': 64,
    'hidden_dims': [128, 64, 32],
    'dropout': 0.2,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'batch_size': 256,
    'epochs': 8,
}

train_loader, val_loader, test_loader = make_metadata_loaders(batch_size=CONFIG['batch_size'])
RANKING_POSITIVE_THRESHOLD = 4.0
RANKING_K = 10
RANKING_NUM_NEGATIVES = 100
user_seen_items = defaultdict(set)
for row in train_encoded[['user_idx', 'item_idx']].itertuples(index=False):
    if int(row.user_idx) != unknown_user_idx and int(row.item_idx) != unknown_item_idx:
        user_seen_items[int(row.user_idx)].add(int(row.item_idx))


In [ ]:
comparison = {}
trained_models = {}

for loss_type in ['mse', 'bpr']:
    model = MetadataMLP(
        num_users,
        num_items,
        CONFIG['embedding_dim'],
        CONFIG['hidden_dims'],
        CONFIG['dropout'],
        categorical_sizes,
        METADATA_NUMERIC_COLS,
    ).to(device)

    best_val_ndcg = train_metadata_mlp(
        model,
        train_loader,
        epochs=CONFIG['epochs'],
        lr=CONFIG['lr'],
        weight_decay=CONFIG['weight_decay'],
        loss_type=loss_type,
    )

    trained_models[loss_type] = model
    val_ranking = evaluate_ranking(model, val_encoded, k=RANKING_K, threshold=RANKING_POSITIVE_THRESHOLD, num_negatives=RANKING_NUM_NEGATIVES)
    test_ranking = evaluate_ranking(model, test_encoded, k=RANKING_K, threshold=RANKING_POSITIVE_THRESHOLD, num_negatives=RANKING_NUM_NEGATIVES)
    val_regression = evaluate_rmse_mae(model, val_loader)
    test_regression = evaluate_rmse_mae(model, test_loader)

    comparison[f'MetadataMLP_{loss_type}'] = {
        'training_mode': loss_type,
        'selection_val_ndcg': best_val_ndcg,
        **{f'val_{k}': v for k, v in val_ranking.items()},
        **{f'test_{k}': v for k, v in test_ranking.items()},
        **{f'val_{k}': v for k, v in val_regression.items()},
        **{f'test_{k}': v for k, v in test_regression.items()},
    }

comparison_df = pd.DataFrame(comparison).T
comparison_df = comparison_df[[
    'training_mode',
    'selection_val_ndcg',
    f'val_hit_rate@{RANKING_K}',
    f'val_ndcg@{RANKING_K}',
    f'val_recall@{RANKING_K}',
    'val_rmse',
    'val_mae',
    f'test_hit_rate@{RANKING_K}',
    f'test_ndcg@{RANKING_K}',
    f'test_recall@{RANKING_K}',
    'test_rmse',
    'test_mae',
]]
comparison_df


MSE epoch 1/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 1/8 - train_loss=0.6552 - val_ndcg@10=0.4939


MSE epoch 2/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 2/8 - train_loss=0.4133 - val_ndcg@10=0.4571


MSE epoch 3/8:   0%|          | 0/4959 [00:00<?, ?it/s]

...

BPR epoch 7/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 7/8 - train_loss=0.0865 - val_ndcg@10=0.6978


BPR epoch 8/8:   0%|          | 0/4959 [00:00<?, ?it/s]

Epoch 8/8 - train_loss=0.0841 - val_ndcg@10=0.7006


,training_mode,selection_val_ndcg,val_hit_rate@10,val_ndcg@10,val_recall@10,val_rmse,val_mae,test_hit_rate@10,test_ndcg@10,test_recall@10,test_rmse,test_mae
MetadataMLP_mse,mse,0.493889,0.790273,0.493889,0.365268,0.593906,0.440799,0.787879,0.483967,0.362306,0.595372,0.455795
MetadataMLP_bpr,bpr,0.700565,0.915777,0.700565,0.582962,3.984626,2.568162,0.896282,0.672853,0.553481,26.812833,26.234326
